# run_fb notebook version

Notebook này là bản chuyển từ `run_fb.py` để chạy tuần tự trong Jupyter.

## Các thay đổi chính
- Bỏ `argparse`, thay bằng một cell cấu hình `CONFIG`
- Dùng `device = torch.device(...)` để chạy linh hoạt trên CPU/GPU
- Giữ nguyên logic train / eval / test
- Chia code thành từng cell dễ debug


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Master/AINC")
# Move to the directory Unifews

MessageError: Error: credential propagation was unsuccessful

In [ ]:

# Cell 1: Imports
import os
import random
import numpy as np
import ptflops

import torch
import torch.nn as nn
import torch.optim as optim

from utils.logger import Logger, ModelLogger, prepare_opt
from utils.loader import load_edgelist
import utils.metric as metric
from archs import identity_n_norm, flops_modules_dict
import archs.models as models

np.set_printoptions(
    linewidth=160,
    edgeitems=5,
    threshold=20,
    formatter=dict(float=lambda x: "% 9.3e" % x),
)
torch.set_printoptions(linewidth=160, edgeitems=5)


In [ ]:

# Cell 2: Notebook config
CONFIG = {
    "seed": 11,
    "dev": 0,              # GPU id; sẽ tự fallback CPU nếu không có CUDA
    "config": "cora",
    "algo": "gcn2_thr",
    "suffix": "",
    "thr_a": 0.0,
    "thr_w": 0.0,
    "layer": 2,
}

# Có thể sửa nhanh ngay trong notebook trước khi chạy các cell dưới
CONFIG


In [ ]:

# Cell 3: Build args từ prepare_opt(parser)
# Ý tưởng: vẫn reuse prepare_opt để lấy toàn bộ default config của project,
# nhưng inject tham số từ notebook thay vì chạy argparse ngoài terminal.

import argparse

parser = argparse.ArgumentParser()
parser.add_argument('-f', '--seed', type=int, default=CONFIG["seed"], help='Random seed.')
parser.add_argument('-v', '--dev', type=int, default=CONFIG["dev"], help='Device id.')
parser.add_argument('-c', '--config', type=str, default=CONFIG["config"], help='Config file name.')
parser.add_argument('-m', '--algo', type=str, default=CONFIG["algo"], help='Model name')
parser.add_argument('-n', '--suffix', type=str, default=CONFIG["suffix"], help='Save name suffix.')
parser.add_argument('-a', '--thr_a', type=float, default=CONFIG["thr_a"], help='Threshold of adj.')
parser.add_argument('-w', '--thr_w', type=float, default=CONFIG["thr_w"], help='Threshold of weight.')
parser.add_argument('-l', '--layer', type=int, default=CONFIG["layer"], help='Layer.')

# Jupyter thường truyền argv riêng; parse_args([]) để không dính lỗi notebook kernel args
args = prepare_opt(parser, argv=[])

# Chọn device an toàn hơn cho notebook
use_cuda = torch.cuda.is_available() and args.dev >= 0
device = torch.device(f"cuda:{args.dev}" if use_cuda else "cpu")

if not ('_' in args.algo):
    args.thr_a, args.thr_w = 0.0, 0.0

print("device:", device)
print("algo:", args.algo)
print("thr_a:", args.thr_a, "thr_w:", args.thr_w)


In [ ]:

# Cell 4: Seed + logger
random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
if use_cuda:
    torch.cuda.manual_seed(args.seed)

flag_run = f"{args.seed}-{args.thr_a:.1e}-{args.thr_w:.1e}"
logger = Logger(args.data, args.algo, flag_run=flag_run)
logger.save_opt(args)

model_logger = ModelLogger(
    logger,
    patience=args.patience,
    cmp='max',
    prefix='model' + args.suffix,
    storage='state_ram' if args.data in ['cs', 'physics', 'arxiv'] else 'state_gpu'
)

stopwatch = metric.Stopwatch()


In [ ]:

# Cell 5: Load data
adj, feat, labels, idx, nfeat, nclass = load_edgelist(
    datastr=args.data,
    datapath=args.path,
    inductive=args.inductive,
    multil=args.multil,
    seed=args.seed
)

print("nfeat =", nfeat)
print("nclass =", nclass)
print("train feat shape =", feat['train'].shape)
print("test feat shape =", feat['test'].shape)


In [ ]:

# Cell 6: Build model
algo_prefix = args.algo.split('_')[0]

if algo_prefix in ['gcn2']:
    model = models.SandwitchThr(
        nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden, nclass=nclass,
        thr_a=args.thr_a, thr_w=args.thr_w, dropout=args.dropout, layer=args.algo
    )
elif algo_prefix in ['mlp']:
    model = models.MLP(
        nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden, nclass=nclass,
        thr_w=args.thr_w, dropout=args.dropout, layer='mlp'
    )
else:
    model = models.GNNThr(
        nlayer=args.layer, nfeat=nfeat, nhidden=args.hidden, nclass=nclass,
        thr_a=args.thr_a, thr_w=args.thr_w, dropout=args.dropout, layer=args.algo
    )

model.reset_parameters()
model.kwargs['diag'] = None
diag = model.kwargs['diag']

adj['train'] = identity_n_norm(
    adj['train'],
    edge_weight=None,
    num_nodes=feat['train'].shape[0],
    rnorm=model.kwargs['rnorm'],
    diag=diag
)

model = model.to(device)
print(model)


In [ ]:

# Cell 7: Optimizer + scheduler + loss
optimizer = optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    threshold=1e-4,
    patience=15
)
loss_fn = nn.BCEWithLogitsLoss() if args.multil else nn.CrossEntropyLoss()


In [ ]:

# Cell 8: Helpers
def move_edge_idx(edge_idx, device):
    if isinstance(edge_idx, tuple):
        return tuple(x.to(device) for x in edge_idx)
    return edge_idx.to(device)


def train_epoch(x, edge_idx, y, idx_split, epoch, verbose=False):
    model.train()
    if epoch < args.epochs // 2:
        model.set_scheme('pruneall', 'pruneall')
    else:
        model.set_scheme('pruneall', 'pruneinc')

    x = x.to(device)
    y = y.to(device)
    edge_idx = move_edge_idx(edge_idx, device)

    stopwatch.reset()
    stopwatch.start()

    optimizer.zero_grad()
    output = model(x, edge_idx, node_lock=torch.tensor([], device=device), verbose=verbose)[idx_split]
    loss = loss_fn(output, y)
    loss.backward()
    optimizer.step()

    stopwatch.pause()
    return loss.item(), stopwatch.time


def eval_epoch(x, edge_idx, y, idx_split, verbose=False):
    model.eval()
    model.set_scheme('keep', 'keep')

    x = x.to(device)
    y = y.to(device)
    edge_idx = move_edge_idx(edge_idx, device)

    calc = metric.F1Calculator(nclass)
    stopwatch.reset()

    with torch.no_grad():
        stopwatch.start()
        output = model(x, edge_idx, node_lock=idx_split, verbose=verbose)[idx_split]
        stopwatch.pause()

        output_cpu = output.detach().cpu()
        ylabel_cpu = y.detach().cpu()

        if args.multil:
            pred = torch.where(
                output_cpu > 0,
                torch.tensor(1, device=output_cpu.device),
                torch.tensor(0, device=output_cpu.device),
            )
        else:
            pred = output_cpu.argmax(dim=1)

        calc.update(ylabel_cpu, pred)
        pred_np = pred.numpy()
        ylabel_np = ylabel_cpu.numpy()

    res = calc.compute(('macro' if args.multil else 'micro'))
    return res, stopwatch.time, pred_np, ylabel_np


def cal_flops(x, edge_idx, idx_split=None, verbose=False):
    model.eval()
    model.set_scheme('keep', 'keep')

    x = x.to(device)
    edge_idx = move_edge_idx(edge_idx, device)

    handle = model.register_forward_hook(models.GNNThr.batch_counter_hook)
    model.__batch_counter_handle__ = handle
    try:
        macs, nparam = ptflops.get_model_complexity_info(
            model, (1, 1, 1),
            input_constructor=lambda _: {'x': x, 'edge_idx': edge_idx},
            custom_modules_hooks=flops_modules_dict,
            as_strings=False,
            print_per_layer_stat=verbose,
            verbose=verbose
        )
    finally:
        handle.remove()

    return macs / 1e9


In [ ]:

# Cell 9: Train
if use_cuda:
    torch.cuda.empty_cache()

time_tol, macs_tol = metric.Accumulator(), metric.Accumulator()
epoch_conv, acc_best = 0, 0

for epoch in range(1, args.epochs + 1):
    verbose = (epoch % 1 == 0) and (logger.lvl_log > 0)

    loss_train, time_epoch = train_epoch(
        x=feat['train'],
        edge_idx=adj['train'],
        y=labels['train'],
        idx_split=idx['train'],
        epoch=epoch,
        verbose=verbose
    )
    time_tol.update(time_epoch)

    acc_val, _, _, _ = eval_epoch(
        x=feat['train'],
        edge_idx=adj['train'],
        y=labels['val'],
        idx_split=idx['val']
    )
    scheduler.step(acc_val)

    macs_epoch = cal_flops(
        x=feat['train'],
        edge_idx=adj['train'],
        idx_split=idx['train']
    )
    macs_tol.update(macs_epoch)

    if verbose:
        res = (
            f"Epoch:{epoch:04d} | "
            f"train loss:{loss_train:.4f}, "
            f"val acc:{acc_val:.4f}, "
            f"time:{time_tol.val:.4f}, "
            f"macs:{macs_tol.val:.4f}"
        )
        if logger.lvl_log > 1:
            logger.print(res)
        else:
            print(res)

    acc_best = model_logger.save_best(acc_val, epoch=epoch)
    if model_logger.is_early_stop(epoch=epoch):
        pass
        # break
    else:
        epoch_conv = max(0, epoch - model_logger.patience)

print("Training finished.")


In [ ]:

# Cell 10: Test
model = model_logger.load('best')
model = model.to(device)

if use_cuda:
    torch.cuda.empty_cache()

adj['test'] = identity_n_norm(
    adj['test'],
    edge_weight=None,
    num_nodes=feat['test'].shape[0],
    rnorm=model.kwargs['rnorm'],
    diag=model.kwargs['diag']
)

acc_test, time_test, outl, labl = eval_epoch(
    x=feat['test'],
    edge_idx=adj['test'],
    y=labels['test'],
    idx_split=idx['test']
)

macs_test = cal_flops(
    x=feat['test'],
    edge_idx=adj['test'],
    idx_split=idx['test']
)

numel_a, numel_w = model.get_numel()

print(f"[Val]  best acc: {acc_best:0.5f} (epoch: {epoch_conv}/{epoch})")
print(f"[Test] best acc: {acc_test:0.5f}")
print(f"[Train] time: {time_tol.val:0.4f} s (avg: {time_tol.avg * 1000:0.1f} ms), MACs: {macs_tol.val:0.3f} G (avg: {macs_tol.avg:0.3f} G)")
print(f"[Test]  time: {time_test:0.4f} s, MACs: {macs_test:0.4f} G, Num adj: {numel_a:0.3f} k, Num weight: {numel_w:0.3f} k")


## Ghi chú khi chạy notebook

1. Chạy notebook từ đúng root của project để import `utils`, `archs` hoạt động.
2. Nếu `prepare_opt(parser, argv=[])` không tương thích với codebase của bạn, đổi thành:
   ```python
   import sys
   sys.argv = [sys.argv[0]]
   args = prepare_opt(parser)
   ```
3. Nếu model của bạn chỉ chạy được trên GPU, giữ `CONFIG["dev"] = 0` và đảm bảo CUDA sẵn sàng.
4. Mình đã sửa `cal_flops()` để `handle.remove()` sau mỗi lần đo, tránh tích lũy hook trong notebook.
